# CrossKEY Demo
**A 3D Cross-modal Keypoint Descriptor for MR-US Matching and Registration**

[Paper (arXiv)](https://arxiv.org/abs/2507.18551) | [Code (GitHub)](https://github.com/morozovdd/CrossKEY)

This notebook trains CrossKEY on Case059 and visualizes 3D keypoint matching between MRI and intraoperative ultrasound.

**Requirements:** Colab with GPU runtime (Runtime > Change runtime type > T4 GPU)

## 1. Setup

Clone the repository, install dependencies, and compile SIFT3D.

In [ ]:
import os

# Clone repository
if not os.path.exists("CrossKEY"):
    !git clone https://github.com/morozovdd/CrossKEY.git

os.chdir("CrossKEY")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install Python dependencies
!pip install -e . -q

# Install system deps and compile SIFT3D
!sudo apt-get update -qq && sudo apt-get install -y -qq zlib1g-dev liblapack-dev libnifti-dev libblas-dev cmake build-essential > /dev/null 2>&1

import os
os.makedirs("external_libs", exist_ok=True)

if not os.path.exists("external_libs/SIFT3D"):
    !git clone https://github.com/morozovdd/SIFT3D.git external_libs/SIFT3D

if not os.path.exists("external_libs/SIFT3D/build/bin/kpSift3D"):
    !cd external_libs/SIFT3D && mkdir -p build && cd build && cmake .. -q && make -j$(nproc) -s

# Create output directories
os.makedirs("data/heatmap", exist_ok=True)
os.makedirs("data/sift_output/mr", exist_ok=True)
os.makedirs("data/sift_output/synthetic_us", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Verify
assert os.path.exists("external_libs/SIFT3D/build/bin/kpSift3D"), "SIFT3D build failed"
print("Setup complete.")

## 2. Preprocessing

Extract SIFT3D keypoints and generate probabilistic heatmaps.

In [ ]:
from pathlib import Path

sift_mr = list(Path("data/sift_output/mr").glob("*_desc.csv"))
sift_us = list(Path("data/sift_output/synthetic_us").glob("*_desc.csv"))
heatmap = list(Path("data/heatmap").glob("main_heatmap.nii.gz"))

if not sift_mr or not sift_us:
    print("Running SIFT3D extraction...")
    !python scripts/run_sift.py --input-dir data/img --output-dir data/sift_output
else:
    print(f"SIFT descriptors found: {len(sift_mr)} MR, {len(sift_us)} synthetic US")

if not heatmap:
    print("Generating heatmaps...")
    !python scripts/create_heatmaps.py --data-dir data/img --output-dir data/heatmap
else:
    print("Heatmaps found")

print("Preprocessing complete.")

## 3. Training

Train the CrossKEY descriptor model on Case059. Takes ~20-30 minutes on a T4 GPU.

In [ ]:
import torch
import yaml
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger, CSVLogger

from src.model.descriptor import Descriptor
from src.data.datamodule import DescriptorDataModule

pl.seed_everything(42)
torch.set_float32_matmul_precision('medium')

# Load config
with open("configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

# Create model
model = Descriptor(
    out_dim=config['model']['out_dim'],
    input_channels=config['model']['input_channels'],
    loss_type=config['loss']['type'],
    margin=config['loss']['margin'],
    temperature=config['loss']['temperature'],
    warmup_epochs=config['loss']['warmup_epochs'],
    spatial_weight=config['loss']['spatial_weight'],
    learning_rate=config['optimizer']['learning_rate'],
    weight_decay=config['optimizer']['weight_decay'],
    max_epochs=config['trainer']['max_epochs'],
    eta_min=config['optimizer']['eta_min'],
)

# Create datamodule
data_config = config['data']
datamodule = DescriptorDataModule(
    data_dir="data",
    batch_size=data_config['batch_size'],
    num_workers=data_config['num_workers'],
    patch_size=(data_config['patch_size'],) * 3,
    num_samples=data_config['num_samples'],
    augment=data_config['augment'],
    max_angle=data_config['max_angle'],
    initial_angle=data_config['initial_angle'],
    angle_warmup_epochs=data_config['angle_warmup_epochs'],
)

print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"Training for {config['trainer']['max_epochs']} epochs")

In [ ]:
# Train
csv_logger = CSVLogger("logs/", name="demo")
tb_logger = TensorBoardLogger("logs/", name="demo_tb")

trainer = pl.Trainer(
    max_epochs=config['trainer']['max_epochs'],
    accelerator="auto",
    devices=1,
    logger=[csv_logger, tb_logger],
    log_every_n_steps=4,
    enable_progress_bar=True,
)

trainer.fit(model, datamodule)
print(f"Training complete. Checkpoint: {trainer.ckpt_path}")

### Training Loss Curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read CSV logs
metrics_file = sorted(Path("logs/demo").glob("version_*/metrics.csv"))[-1]
df = pd.read_csv(metrics_file)
loss = df.dropna(subset=["train/loss"])

plt.figure(figsize=(10, 4))
plt.plot(loss["epoch"], loss["train/loss"], linewidth=0.8)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CrossKEY Training Loss (Case059)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Evaluation

Load the trained model and run keypoint matching between MR and real intraoperative ultrasound.

In [ ]:
import numpy as np
from src.model.matcher import KNNMatcher

# Load trained model
ckpt_dir = sorted(Path("logs/demo_tb").glob("version_*/checkpoints/*.ckpt"))
if ckpt_dir:
    checkpoint_path = str(ckpt_dir[-1])
else:
    # Fallback: use Lightning's last checkpoint
    checkpoint_path = trainer.ckpt_path

print(f"Loading checkpoint: {checkpoint_path}")
model = Descriptor.load_from_checkpoint(checkpoint_path)
model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")
device = next(model.parameters()).device

# Setup inference data
eval_dm = DescriptorDataModule(
    data_dir="data",
    batch_size=64,
    num_workers=2,
    patch_size=(32, 32, 32),
    grid_spacing=8,
)
eval_dm.setup(stage="test")

# Extract descriptors
all_desc, all_pts, all_mod = [], [], []
with torch.no_grad():
    for batch in eval_dm.test_dataloader():
        desc = model(batch["patch"].to(device))
        all_desc.append(desc.cpu())
        all_pts.append(batch["point"].cpu())
        all_mod.extend(batch["modality"])

all_desc = torch.cat(all_desc)
all_pts = torch.cat(all_pts)
mr_mask = torch.tensor([m == "mr" for m in all_mod])

mr_desc, us_desc = all_desc[mr_mask], all_desc[~mr_mask]
mr_pts, us_pts = all_pts[mr_mask], all_pts[~mr_mask]

print(f"Descriptors: {len(mr_desc)} MR, {len(us_desc)} US")

# Match
matcher = KNNMatcher(
    k=2, distance_threshold=float("inf"), ratio_threshold=0.75,
    mutual=True, metric="euclidean", evaluation_threshold=5.0,
)
match_pairs, metrics = matcher.match_and_evaluate(mr_desc, us_desc, mr_pts, us_pts)

print(f"\nResults:")
print(f"  Matches: {metrics['num_matches']}")
print(f"  Correct: {metrics['num_correct']}")
print(f"  Precision: {metrics['precision']:.1f}%")
print(f"  Matching score: {metrics['matching_score']:.4f}")

## 5. 3D Visualization

Interactive 3D scatter plot of matched keypoints between MR and US volumes.

In [ ]:
import plotly.graph_objects as go

mr_pts_np = mr_pts.numpy()
us_pts_np = us_pts.numpy()

# Extract match indices
match_src = np.array([p[0] for p in match_pairs])
match_tgt = np.array([p[1] for p in match_pairs])
match_dist = np.array([p[2] for p in match_pairs])

# Classify correct/incorrect
mr_matched = mr_pts_np[match_src]
us_matched = us_pts_np[match_tgt]
spatial_dist = np.linalg.norm(mr_matched - us_matched, axis=1)
correct = spatial_dist < 5.0

fig = go.Figure()

# All keypoints (faded)
fig.add_trace(go.Scatter3d(
    x=mr_pts_np[:, 0], y=mr_pts_np[:, 1], z=mr_pts_np[:, 2],
    mode="markers", marker=dict(size=1.5, color="rgba(65,105,225,0.2)"),
    name=f"MR keypoints ({len(mr_pts_np)})",
))
fig.add_trace(go.Scatter3d(
    x=us_pts_np[:, 0], y=us_pts_np[:, 1], z=us_pts_np[:, 2],
    mode="markers", marker=dict(size=1.5, color="rgba(220,20,60,0.2)"),
    name=f"US keypoints ({len(us_pts_np)})",
))

# Correct matches
if correct.any():
    c_mr = mr_matched[correct]
    c_us = us_matched[correct]
    fig.add_trace(go.Scatter3d(
        x=c_mr[:, 0], y=c_mr[:, 1], z=c_mr[:, 2],
        mode="markers", marker=dict(size=5, color="royalblue"),
        name=f"MR correct ({correct.sum()})",
    ))
    fig.add_trace(go.Scatter3d(
        x=c_us[:, 0], y=c_us[:, 1], z=c_us[:, 2],
        mode="markers", marker=dict(size=5, color="crimson"),
        name=f"US correct ({correct.sum()})",
    ))
    # Lines for correct matches
    lx, ly, lz = [], [], []
    for i in np.where(correct)[0]:
        lx.extend([mr_matched[i, 0], us_matched[i, 0], None])
        ly.extend([mr_matched[i, 1], us_matched[i, 1], None])
        lz.extend([mr_matched[i, 2], us_matched[i, 2], None])
    fig.add_trace(go.Scatter3d(
        x=lx, y=ly, z=lz, mode="lines",
        line=dict(color="rgba(0,200,0,0.6)", width=2),
        name="Correct matches",
    ))

# Incorrect matches
if (~correct).any():
    ic_mr = mr_matched[~correct]
    ic_us = us_matched[~correct]
    lx, ly, lz = [], [], []
    for i in np.where(~correct)[0]:
        lx.extend([mr_matched[i, 0], us_matched[i, 0], None])
        ly.extend([mr_matched[i, 1], us_matched[i, 1], None])
        lz.extend([mr_matched[i, 2], us_matched[i, 2], None])
    fig.add_trace(go.Scatter3d(
        x=lx, y=ly, z=lz, mode="lines",
        line=dict(color="rgba(255,0,0,0.3)", width=1),
        name=f"Incorrect ({(~correct).sum()})",
    ))

fig.update_layout(
    title=f"CrossKEY Matching -- {metrics['num_matches']} matches, {metrics['precision']:.1f}% precision",
    scene=dict(xaxis_title="H", yaxis_title="W", zaxis_title="D", aspectmode="data"),
    width=900, height=650,
)
fig.show()

## 6. Slice Visualization

Matched keypoints overlaid on axial MR and US slices.

In [ ]:
from src.utils.utils import load_nifti

mr_vol = load_nifti("data/img/mr/Case059-t2_full.nii.gz")
us_vol = load_nifti("data/img/us/Case059-us_full.nii.gz")
mr_norm = (mr_vol - mr_vol.min()) / (mr_vol.max() - mr_vol.min() + 1e-8)
us_norm = (us_vol - us_vol.min()) / (us_vol.max() - us_vol.min() + 1e-8)

# Unpad matched points (padding = patch_size // 2 = 16)
pad = 16
mr_match_orig = mr_matched - pad
us_match_orig = us_matched - pad

# Plot 3 axial slices
z_slices = [mr_vol.shape[2] // 4, mr_vol.shape[2] // 2, 3 * mr_vol.shape[2] // 4]
fig, axes = plt.subplots(len(z_slices), 2, figsize=(14, 5 * len(z_slices)))

for row, z in enumerate(z_slices):
    axes[row, 0].imshow(mr_norm[:, :, z], cmap="gray", origin="lower")
    axes[row, 0].set_title(f"MR (z={z})")
    axes[row, 1].imshow(us_norm[:, :, z], cmap="gray", origin="lower")
    axes[row, 1].set_title(f"US (z={z})")

    # Show matches near this slice
    z_range = 10
    for i in range(len(mr_match_orig)):
        if abs(mr_match_orig[i, 2] - z) < z_range:
            color = "lime" if correct[i] else "red"
            axes[row, 0].plot(mr_match_orig[i, 1], mr_match_orig[i, 0], "o",
                              color=color, markersize=6, markeredgecolor="white", markeredgewidth=0.5)
            axes[row, 1].plot(us_match_orig[i, 1], us_match_orig[i, 0], "o",
                              color=color, markersize=6, markeredgecolor="white", markeredgewidth=0.5)

    for ax in axes[row]:
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle(f"CrossKEY Matching: {metrics['num_matches']} matches, {metrics['precision']:.1f}% precision\n"
             f"Green = correct (<5mm), Red = incorrect", fontsize=13)
plt.tight_layout()
plt.show()

## Optional: Save Checkpoint to Google Drive

In [ ]:
# Uncomment to save checkpoint to Google Drive
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy(checkpoint_path, "/content/drive/MyDrive/crosskey_case059.ckpt")
# print("Checkpoint saved to Google Drive")